# MapleStory Idle AOS — D3 ROAS Weekly Trend

| | |
|---|---|
| **Bundle** | `com.nexon.ma` |
| **OS** | Android |
| **Period** | Last 90 days (rolling from run date) |
| **Primary KPI** | D3 ROAS · Target: **25%** (source: Nexon product one-pager) |
| **Table** | `moloco-ae-view.market_share.fact_market` |
| **Author** | Haewon Yum · KOR GDS |
| **Created** | 2026-04-29 |

In [ ]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

client = bigquery.Client(project='moloco-ods')

def run_query(query, label=''):
    df = client.query(query).result().to_dataframe()
    print(f'✅ {label}: {len(df)} rows')
    return df

BUNDLE    = 'com.nexon.ma'
OS        = 'ANDROID'
D3_TARGET = 25.0  # Nexon-stated D3 ROAS target (%)

## Section 1 — Overall Weekly D3 ROAS

In [ ]:
Q_OVERALL = """
SELECT
  DATE_TRUNC(install_date_utc, WEEK(MONDAY))                             AS week_start,
  SUM(moloco.gross_spend_usd)                                            AS spend_usd,
  SUM(moloco.installs)                                                   AS installs,
  SAFE_DIVIDE(SUM(moloco.kpi_revenue_d3), SUM(moloco.gross_spend_usd))   AS d3_roas,
  SAFE_DIVIDE(SUM(moloco.kpi_revenue_d7), SUM(moloco.gross_spend_usd))   AS d7_roas
FROM `moloco-ae-view.market_share.fact_market`
WHERE
  app_market_bundle = '{bundle}'
  AND os             = '{os}'
  AND install_date_utc BETWEEN DATE_SUB(CURRENT_DATE(), INTERVAL 90 DAY)
                           AND CURRENT_DATE()
GROUP BY 1
ORDER BY 1
""".format(bundle=BUNDLE, os=OS)

df_overall = run_query(Q_OVERALL, 'Overall weekly')
df_overall['week_start']   = pd.to_datetime(df_overall['week_start'])
df_overall['d3_roas_pct']  = df_overall['d3_roas'] * 100
df_overall['d7_roas_pct']  = df_overall['d7_roas'] * 100
df_overall

## Section 2 — Weekly D3 ROAS by Geo

In [ ]:
Q_GEO = """
WITH base AS (
  SELECT
    DATE_TRUNC(install_date_utc, WEEK(MONDAY)) AS week_start,
    CASE
      WHEN country IN ('KOR','TWN','USA','THA','CAN','SGP','MYS') THEN country
      ELSE 'Other'
    END AS geo,
    SUM(moloco.kpi_revenue_d3)  AS rev_d3,
    SUM(moloco.gross_spend_usd) AS spend_usd
  FROM `moloco-ae-view.market_share.fact_market`
  WHERE
    app_market_bundle = '{bundle}'
    AND os = '{os}'
    AND install_date_utc BETWEEN DATE_SUB(CURRENT_DATE(), INTERVAL 90 DAY)
                             AND CURRENT_DATE()
  GROUP BY 1, 2
)
SELECT
  week_start,
  geo,
  SAFE_DIVIDE(rev_d3, spend_usd) AS d3_roas,
  spend_usd
FROM base
ORDER BY 1, 2
""".format(bundle=BUNDLE, os=OS)

df_geo = run_query(Q_GEO, 'Geo weekly')
df_geo['week_start']  = pd.to_datetime(df_geo['week_start'])
df_geo['d3_roas_pct'] = df_geo['d3_roas'] * 100

# Pivot to wide format
df_pivot = df_geo.pivot_table(
    index='week_start', columns='geo', values='d3_roas_pct', aggfunc='first'
)
df_pivot.round(1)

## Section 3 — Charts

In [ ]:
# ── Chart 1: Overall D3 ROAS weekly trend ──────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))

# Separate complete vs partial weeks (last week is partial — D3 cohort not matured)
df_complete = df_overall.iloc[:-1]
df_partial  = df_overall.iloc[-1:]

ax.plot(df_complete['week_start'], df_complete['d3_roas_pct'],
        color='#4f46e5', linewidth=2.5, marker='o', markersize=5, label='D3 ROAS')
ax.fill_between(df_complete['week_start'], df_complete['d3_roas_pct'],
                alpha=0.10, color='#4f46e5')

# Partial week (dashed)
ax.plot([df_complete['week_start'].iloc[-1], df_partial['week_start'].iloc[0]],
        [df_complete['d3_roas_pct'].iloc[-1], df_partial['d3_roas_pct'].iloc[0]],
        color='#4f46e5', linewidth=1.5, linestyle=':', marker='o', markersize=4)
ax.annotate('partial\n(D3 pending)', xy=(df_partial['week_start'].iloc[0],
            df_partial['d3_roas_pct'].iloc[0]),
            xytext=(8, 8), textcoords='offset points', fontsize=8, color='#9ca3af')

# Target line
ax.axhline(D3_TARGET, color='#ef4444', linestyle='--', linewidth=1.5,
           label=f'Nexon target ({D3_TARGET:.0f}%)')
ax.text(df_overall['week_start'].min(), D3_TARGET + 1.5, 'Target 25%',
        color='#ef4444', fontsize=8)

ax.set_title('MapleStory Idle (AOS) — Overall Weekly D3 ROAS', fontsize=14,
             fontweight='bold', pad=12)
ax.set_ylabel('D3 ROAS (%)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
plt.xticks(rotation=30, ha='right')
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.35)
ax.set_facecolor('#f9fafb')
fig.patch.set_facecolor('white')
fig.tight_layout()
plt.savefig('overall_d3roas_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: overall_d3roas_trend.png')

In [ ]:
# ── Chart 2: D3 ROAS by geo (weekly) ──────────────────────────────────────
GEO_COLORS = {
    'KOR': '#4f46e5',
    'USA': '#f59e0b',
    'THA': '#10b981',
    'TWN': '#ef4444',
    'SGP': '#8b5cf6',
    'MYS': '#06b6d4',
}
FOCUS_GEOS = ['KOR', 'USA', 'THA', 'TWN']  # highest spend + most signal

fig, ax = plt.subplots(figsize=(13, 5))

for geo in FOCUS_GEOS:
    if geo not in df_pivot.columns:
        continue
    series = df_pivot[geo].dropna()
    ax.plot(series.index, series.values,
            color=GEO_COLORS.get(geo, '#6b7280'),
            linewidth=2, marker='o', markersize=4, label=geo)

# Target line
ax.axhline(D3_TARGET, color='#374151', linestyle='--', linewidth=1.2,
           alpha=0.6, label=f'Target ({D3_TARGET:.0f}%)')

ax.set_title('MapleStory Idle (AOS) — Weekly D3 ROAS by Geo', fontsize=14,
             fontweight='bold', pad=12)
ax.set_ylabel('D3 ROAS (%)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
plt.xticks(rotation=30, ha='right')
ax.legend(fontsize=10, loc='upper right')
ax.grid(axis='y', linestyle='--', alpha=0.35)
ax.set_facecolor('#f9fafb')
fig.patch.set_facecolor('white')
fig.tight_layout()
plt.savefig('geo_d3roas_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: geo_d3roas_trend.png')

## Key Observations

**Target: D3 ROAS 25%** (Nexon-stated; source: MapleStory_Idle_AOS_GL_Product_Team_OnePager)

*(To be filled after chart review.)*